# Silver: Airports

Filter to relevant airports + standardize fields.

## Import and Setup

In [0]:
email = dbutils.secrets.get(scope="flight-delay", key="db-email")

In [0]:
import sys
sys.path.insert(0, f"/Workspace/Users/{email}/ml-final-project/databricks")

from common.config import (
    BRONZE_AIRPORTS_PATH,
    abfss, SILVER_CONTAINER,
    configure_storage_auth,
)

from pyspark.sql import functions as F

SILVER_AIRPORTS_PATH = abfss(SILVER_CONTAINER, "airports")

configure_storage_auth(spark)

## Read From Bronze

In [0]:
bronze = spark.read.format("delta").load(BRONZE_AIRPORTS_PATH)

## Clean and Filter

 - Keep only airports with a valid IATA code (we join on IATA)
 - Keep only scheduled-service airports (commercial passenger flights)
 - Keep only reasonable types (exclude heliports, closed airports, seaplane bases)

In [0]:
silver = (
    bronze
    .filter(F.col("iata_code").isNotNull())
    .filter(F.col("iata_code") != "")
    .filter(F.col("type").isin("large_airport", "medium_airport", "small_airport"))
    .filter(F.col("scheduled_service") == "yes")
    .select(
        F.upper(F.col("iata_code")).alias("iata_code"),
        F.upper(F.col("ident")).alias("icao_code"),
        F.col("type").alias("airport_type"),  # large/medium/small
        F.col("name").alias("airport_name"),
        F.col("latitude_deg").cast("double").alias("latitude"),
        F.col("longitude_deg").cast("double").alias("longitude"),
        F.col("elevation_ft").cast("int").alias("elevation_ft"),
        F.col("continent"),
        F.col("iso_country").alias("country_code"),
        F.col("iso_region").alias("region_code"),
        F.col("municipality").alias("city"),
    )
    # Size encoded as a numeric feature
    .withColumn(
        "airport_size_rank",
        F.when(F.col("airport_type") == "large_airport", 3)
         .when(F.col("airport_type") == "medium_airport", 2)
         .otherwise(1)
    )
    .withColumn("silver_processed_ts_utc", F.current_timestamp())
)

## Deduplicate by IATA code
Some IATA codes appear more than once (e.g., moved airports).
Keep the largest airport for each code.

In [0]:

from pyspark.sql.window import Window

w = Window.partitionBy("iata_code").orderBy(F.desc("airport_size_rank"))

silver_deduped = (
    silver
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print(f"Silver airports: {silver_deduped.count():,}")

## Write to Silver

In [0]:
(
    silver_deduped.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_AIRPORTS_PATH)
)

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS silver.airports
    USING DELTA
    LOCATION '{SILVER_AIRPORTS_PATH}'
""")

## Exit

In [0]:
dbutils.notebook.exit(f'{{"rows_in_silver": {silver_deduped.count()}}}')